# Unsupervised Learning

In supervised learning, the algorithm learns from labeled examples — data paired with correct answers. **Unsupervised learning** works without labels. The algorithm receives raw data and must discover **hidden structure** on its own: natural groupings, underlying patterns, compressed representations, or unusual observations.

Unsupervised learning answers fundamentally different questions. Are there distinct customer segments in purchase data? Can we visualize a 500-dimensional dataset in two dimensions without losing essential information? Which transactions deviate so far from normal behavior that they might be fraudulent? No human has annotated the answers — the algorithm must find them.

This notebook covers the major unsupervised learning techniques: clustering, dimensionality reduction, and anomaly detection. For each, we examine how the algorithm works, when to use it, how to evaluate results without labels, and practical code demonstrations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. The Unsupervised Learning Framework

Unsupervised learning receives a dataset $\{x_1, x_2, \ldots, x_n\}$ without target labels. The goal is to find useful structure:

- **Clustering** — partition data into groups of similar points.
- **Dimensionality reduction** — represent data in fewer dimensions while preserving information.
- **Anomaly detection** — identify points that do not fit the normal pattern.
- **Density estimation** — model the probability distribution of the data.
- **Association rule mining** — discover relationships between variables (e.g., customers who buy bread also buy butter).

Unlike supervised learning, there is no single correct answer to compare against. Evaluation is indirect: internal metrics (silhouette score, inertia), visual inspection, or downstream task performance (does clustering improve a later supervised model?).

Unsupervised learning is used for **exploratory data analysis** — understanding data before building predictive models — and as a preprocessing step that creates features for supervised learning.

---

## 2. Clustering

Clustering groups data points so that members of the same cluster are more similar to each other than to members of other clusters. The algorithm does not know cluster names or how many clusters exist — it discovers structure from geometry alone.

Formally, clustering partitions $n$ points into $K$ disjoint sets $C_1, C_2, \ldots, C_K$ such that within-cluster similarity is maximized and between-cluster similarity is minimized.

**Applications:** customer segmentation, document grouping, image segmentation, gene expression analysis, social network community detection.

### 2.1 K-Means Clustering

K-Means is the most widely used clustering algorithm. Given $K$ (the number of clusters), it iteratively assigns each point to the nearest cluster center (centroid) and recomputes centroids as the mean of assigned points.

**Algorithm:**

1. Initialize $K$ centroids randomly (or using K-Means++).
2. **Assign** each point to the nearest centroid.
3. **Update** each centroid to the mean of its assigned points.
4. Repeat steps 2–3 until centroids stop moving (convergence).

K-Means minimizes **inertia** (within-cluster sum of squares):

$$\text{Inertia} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

where $\mu_k$ is the centroid of cluster $C_k$.

**Strengths:** Fast, scales well to large datasets, simple to understand.

**Limitations:** Must specify $K$ in advance. Assumes spherical, equally sized clusters. Sensitive to initialization and outliers. Struggles with non-convex shapes.

In [ ]:
# K-Means clustering
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
axes[0].set_title("True labels (unknown to algorithm)")

axes[1].scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c="red", marker="X", s=200, linewidths=2, label="Centroids")
axes[1].set_title(f"K-Means (K=4, silhouette={silhouette_score(X, labels):.3f})")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

### 2.2 Choosing the Number of Clusters

When $K$ is unknown, several methods help select it:

**Elbow method** — plot inertia vs $K$. Inertia always decreases as $K$ increases (each point becomes its own cluster at $K = n$). Look for the "elbow" where adding more clusters yields diminishing returns.

**Silhouette score** — for each point, measures how similar it is to its own cluster compared to the nearest other cluster. Range $[-1, 1]$; higher is better. Average across all points gives an overall score. Choose $K$ with the highest silhouette.

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

where $a(i)$ is mean distance to points in the same cluster and $b(i)$ is mean distance to the nearest other cluster.

**Domain knowledge** — sometimes the number of clusters is dictated by the problem (3 customer tiers, 5 product categories).

In [ ]:
# Elbow method and silhouette analysis
K_range = range(2, 11)
inertias, silhouettes = [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, "bo-", linewidth=2)
axes[0].set_xlabel("Number of clusters K")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(K_range, silhouettes, "ro-", linewidth=2)
best_k = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(best_k, color="green", linestyle="--", label=f"Best K={best_k}")
axes[1].set_xlabel("Number of clusters K")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Best K by silhouette: {best_k} (score={max(silhouettes):.3f})")

### 2.3 Hierarchical Clustering

Hierarchical clustering builds a **tree of clusters** (dendrogram) rather than a flat partition. Two approaches:

**Agglomerative (bottom-up)** — start with each point as its own cluster. Repeatedly merge the two closest clusters until one cluster remains.

**Divisive (top-down)** — start with all points in one cluster. Repeatedly split clusters until each point is alone.

Agglomerative is more common. The key choice is the **linkage criterion** — how to measure distance between clusters:

- **Single linkage** — distance between closest points in two clusters. Can produce elongated chains.
- **Complete linkage** — distance between farthest points. Produces compact clusters.
- **Average linkage** — average distance between all pairs across clusters.
- **Ward linkage** — minimizes increase in total within-cluster variance. Often produces the most balanced results.

The dendrogram shows the merge history. Cut the tree at a chosen height to get a specific number of clusters. Unlike K-Means, hierarchical clustering does not require specifying $K$ upfront — you choose it by cutting the dendrogram.

In [ ]:
# Hierarchical clustering dendrogram
X_small = X[:60]  # subset for readable dendrogram
Z = linkage(X_small, method="ward")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dendrogram(Z, ax=axes[0], truncate_mode="lastp", p=8)
axes[0].set_title("Dendrogram (Ward linkage)")
axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("Distance")

hc = AgglomerativeClustering(n_clusters=4, linkage="ward")
hc_labels = hc.fit_predict(X)
axes[1].scatter(X[:, 0], X[:, 1], c=hc_labels, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
axes[1].set_title(f"Agglomerative Clustering (K=4, silhouette={silhouette_score(X, hc_labels):.3f})")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

### 2.4 DBSCAN

**Density-Based Spatial Clustering of Applications with Noise (DBSCAN)** groups points that are closely packed and marks points in low-density regions as **outliers**.

Key parameters:
- **eps** ($\varepsilon$) — maximum distance between two points to be considered neighbors.
- **min_samples** — minimum number of points in a neighborhood to form a core point.

Point types:
- **Core point** — has at least `min_samples` neighbors within distance `eps`.
- **Border point** — within `eps` of a core point but not a core point itself.
- **Noise point** — neither core nor border. Treated as outlier.

**Strengths:** Does not require specifying $K$. Finds clusters of arbitrary shape. Naturally identifies outliers.

**Limitations:** Sensitive to `eps` and `min_samples`. Struggles with clusters of varying density.

In [ ]:
# DBSCAN on non-spherical clusters
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.labels_, cmap="coolwarm", alpha=0.7, edgecolors="k", linewidth=0.3)
axes[0].set_title("K-Means (fails on non-spherical)")

db = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db.labels_, cmap="coolwarm", alpha=0.7, edgecolors="k", linewidth=0.3)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
axes[1].set_title(f"DBSCAN (eps=0.2, {n_clusters} clusters)")

db2 = DBSCAN(eps=0.15, min_samples=5).fit(X_moons)
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=db2.labels_, cmap="coolwarm", alpha=0.7, edgecolors="k", linewidth=0.3)
axes[2].set_title("DBSCAN (eps=0.15, too strict → noise)")

for ax in axes:
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

### 2.5 Gaussian Mixture Models (GMM)

GMM assumes data is generated from a mixture of $K$ Gaussian distributions. Each cluster is a Gaussian with its own mean and covariance. The algorithm uses **Expectation-Maximization (EM)** to estimate parameters:

1. **E-step** — compute the probability that each point belongs to each Gaussian.
2. **M-step** — update Gaussian parameters (mean, covariance, weight) based on assignments.
3. Repeat until convergence.

Unlike K-Means (hard assignment — each point belongs to exactly one cluster), GMM provides **soft assignment** — each point has a probability of belonging to each cluster.

GMM handles elliptical clusters (K-Means assumes spherical) and provides probabilistic cluster membership. The **Bayesian Information Criterion (BIC)** or **Akaike Information Criterion (AIC)** can select the number of components.

In [ ]:
# GMM with soft assignments
gmm = GaussianMixture(n_components=4, random_state=42).fit(X)
gmm_labels = gmm.predict(X)
probs = gmm.predict_proba(X)

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=gmm_labels, cmap="viridis", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.title(f"GMM (K=4, silhouette={silhouette_score(X, gmm_labels):.3f})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

print("Soft assignments for first 5 points (prob of each cluster):")
print(np.round(probs[:5], 3))

---

## 3. Dimensionality Reduction

High-dimensional data is difficult to visualize, expensive to compute on, and prone to overfitting (the **curse of dimensionality** — in high dimensions, all points become roughly equidistant, and distances lose meaning). Dimensionality reduction projects data into fewer dimensions while preserving structure.

**Applications:** data visualization, noise removal, feature extraction before supervised learning, data compression, speeding up training.

### 3.1 Principal Component Analysis (PCA)

PCA finds new axes (principal components) that capture the maximum variance in the data. The first principal component is the direction of greatest spread; the second is orthogonal to the first and captures the next most variance, and so on.

Mathematically, PCA computes the **eigenvectors** of the data's covariance matrix. The eigenvector with the largest eigenvalue is the first principal component.

$$\text{Covariance matrix:} \quad \mathbf{C} = \frac{1}{n} \mathbf{X}^T \mathbf{X}$$

Projecting data onto the top $k$ principal components gives a $k$-dimensional representation that minimizes reconstruction error.

**Explained variance ratio** — the proportion of total variance captured by each component. If the first 2 components capture 95% of variance, reducing to 2 dimensions loses little information.

PCA is linear — it cannot capture nonlinear relationships. It works best when variance corresponds to meaningful structure.

In [ ]:
# PCA on digits dataset (64 features → 2D)
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_digits)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=digits.target, cmap="tab10", alpha=0.6, s=10)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
axes[0].set_title("PCA: 64D digits → 2D")
plt.colorbar(scatter, ax=axes[0], label="Digit")

pca_full = PCA().fit(X_digits)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumvar)+1), cumvar, "b-o", markersize=3)
axes[1].axhline(0.95, color="r", linestyle="--", label="95% variance")
n_95 = np.argmax(cumvar >= 0.95) + 1
axes[1].axvline(n_95, color="g", linestyle=":", label=f"{n_95} components")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative explained variance")
axes[1].set_title("Scree Plot")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"2 components explain {pca.explained_variance_ratio_.sum():.1%} of variance")
print(f"{n_95} components needed for 95% variance")

### 3.2 t-SNE (t-Distributed Stochastic Neighbor Embedding)

t-SNE is a nonlinear dimensionality reduction technique designed for **visualization**. It maps high-dimensional data to 2D or 3D while preserving local neighborhood structure — points that are close in high dimensions stay close in the visualization.

t-SNE models pairwise similarities in high and low dimensions using probability distributions, then minimizes the difference (KL divergence) between them. It uses a t-distribution in the low-dimensional space to prevent the "crowding problem" where many points collapse to the center.

**Strengths:** Excellent for visualizing cluster structure. Reveals patterns invisible to linear methods.

**Limitations:** Slow on large datasets. Non-deterministic (results vary between runs). Distances between clusters in the plot are not meaningful — only local neighborhoods are preserved. Not suitable as a preprocessing step for downstream models (use PCA for that).

In [ ]:
# PCA vs t-SNE visualization
X_sub = X_digits[:500]
y_sub = digits.target[:500]

X_pca_sub = PCA(n_components=2).fit_transform(X_sub)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(X_pca_sub[:, 0], X_pca_sub[:, 1], c=y_sub, cmap="tab10", alpha=0.6, s=15)
axes[0].set_title("PCA (linear)")

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap="tab10", alpha=0.6, s=15)
axes[1].set_title("t-SNE (nonlinear)")

for ax in axes:
    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")

plt.tight_layout()
plt.show()

### 3.3 Other Dimensionality Reduction Methods

**Independent Component Analysis (ICA)** — separates data into statistically independent components. Used in signal processing (separating mixed audio sources) and fMRI analysis.

**Autoencoders** — neural networks that compress input to a low-dimensional bottleneck and reconstruct it. The bottleneck layer serves as a learned dimensionality reduction. Powerful for nonlinear structure but requires more data and tuning.

**UMAP (Uniform Manifold Approximation and Projection)** — similar to t-SNE but faster and better preserves global structure. Increasingly popular for visualization.

The choice depends on purpose: PCA for preprocessing and variance preservation, t-SNE/UMAP for visualization, autoencoders for complex nonlinear compression.

---

## 4. Anomaly Detection

Anomaly detection (outlier detection) identifies data points that deviate significantly from the majority. Unlike clustering (which groups normal data), anomaly detection focuses on the rare unusual points.

**Applications:** fraud detection, network intrusion detection, manufacturing defect detection, medical abnormality flagging, equipment failure prediction.

Anomalies fall into types:
- **Point anomalies** — individual data points that are unusual (a \$10,000 purchase when typical is \$50).
- **Contextual anomalies** — unusual in a specific context (80°F is normal in summer but anomalous in winter).
- **Collective anomalies** — a collection of points that together are anomalous (a coordinated fraud pattern).

### 4.1 Statistical Methods

The simplest approach: model the normal distribution and flag points beyond a threshold.

- **Z-score** — flag points where $|z| = \frac{|x - \mu|}{\sigma} > 3$.
- **IQR method** — flag points below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$.

These assume univariate or approximately normal data. They are fast and interpretable but do not capture multivariate relationships.

### 4.2 Distance-Based Methods

**K-Nearest Neighbors distance** — a point is anomalous if the average distance to its $K$ nearest neighbors is unusually large. Normal points are in dense regions; anomalies are isolated.

**Local Outlier Factor (LOF)** — compares the local density of a point to its neighbors. A point in a sparse region surrounded by dense regions has a high LOF score.

### 4.3 Isolation Forest

Isolation Forest builds random trees that recursively split data on random features and random thresholds. **Anomalies are easier to isolate** — they require fewer splits to separate from the rest. The average path length to isolate a point serves as its anomaly score.

Short path length → anomaly. Long path length → normal.

Isolation Forest is efficient (linear time), works in high dimensions, and requires no distance metric. It is one of the most practical anomaly detection algorithms.

### 4.4 One-Class SVM

One-Class SVM learns a boundary around normal data in feature space (or kernel space). Points outside the boundary are anomalies. It is related to standard SVM but trained only on normal (inlier) data.

Effective when normal data is well-defined and anomalies are rare. Sensitive to hyperparameters and less scalable than Isolation Forest.

In [ ]:
# Anomaly detection comparison
X_normal, _ = make_blobs(n_samples=300, centers=1, cluster_std=0.5, random_state=42)
rng = np.random.RandomState(42)
X_anomalies = rng.uniform(low=-6, high=6, size=(20, 2))
X_all = np.vstack([X_normal, X_anomalies])
true_labels = np.array([1]*300 + [-1]*20)  # 1=normal, -1=anomaly

iso = IsolationForest(contamination=0.06, random_state=42).fit(X_all)
ocsvm = OneClassSVM(nu=0.06, kernel="rbf", gamma=0.5).fit(X_all)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, name in zip(axes, [iso, ocsvm], ["Isolation Forest", "One-Class SVM"]):
    preds = model.predict(X_all)
    ax.scatter(X_all[preds==1, 0], X_all[preds==1, 1], c="blue", alpha=0.5, s=20, label="Normal")
    ax.scatter(X_all[preds==-1, 0], X_all[preds==-1, 1], c="red", marker="x", s=60, label="Anomaly")
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()

---

## 5. Association Rule Mining

Association rule mining discovers relationships between items in transactional data. The classic example: *customers who buy diapers also buy beer.*

Given transactions (baskets of items), find rules of the form $\{A, B\} \rightarrow \{C\}$ — if a customer buys A and B, they are likely to also buy C.

Key metrics:
- **Support** — frequency of itemset in all transactions. $\text{support}(A, B) = \frac{\text{count}(A \cap B)}{N}$
- **Confidence** — how often the rule is true. $\text{confidence}(A \rightarrow B) = \frac{\text{support}(A, B)}{\text{support}(A)}$
- **Lift** — how much more likely B is when A is present. $\text{lift}(A \rightarrow B) = \frac{\text{confidence}(A \rightarrow B)}{\text{support}(B)}$. Lift > 1 means positive association.

The **Apriori algorithm** efficiently finds frequent itemsets by pruning candidates that cannot meet the minimum support threshold.

Applications: market basket analysis, recommendation systems, web usage mining, medical symptom co-occurrence.

In [ ]:
# Simple association rule demonstration
transactions = [
    ["bread", "milk", "eggs"],
    ["bread", "butter"],
    ["milk", "butter", "eggs"],
    ["bread", "milk", "butter"],
    ["bread", "milk"],
    ["milk", "eggs"],
    ["bread", "milk", "butter", "eggs"],
    ["bread", "butter"],
]

N = len(transactions)

def support(itemset):
    return sum(1 for t in transactions if all(i in t for i in itemset)) / N

sup_bread = support(["bread"])
sup_milk = support(["milk"])
sup_bread_milk = support(["bread", "milk"])
confidence = sup_bread_milk / sup_bread
lift = confidence / sup_milk

print(f"support(bread)       = {sup_bread:.3f}")
print(f"support(milk)        = {sup_milk:.3f}")
print(f"support(bread, milk) = {sup_bread_milk:.3f}")
print(f"confidence(bread → milk) = {confidence:.3f}")
print(f"lift(bread → milk)       = {lift:.3f}")

---

## 6. Evaluating Unsupervised Learning

Without labels, evaluation is inherently subjective. Common approaches:

### 6.1 Internal Metrics (no labels needed)

- **Silhouette score** — cohesion vs separation. Higher is better.
- **Calinski-Harabasz index** — ratio of between-cluster to within-cluster dispersion. Higher is better.
- **Davies-Bouldin index** — average similarity between each cluster and its most similar cluster. Lower is better.
- **Inertia** (K-Means) — within-cluster sum of squares. Lower is better but always decreases with more clusters.
- **Explained variance** (PCA) — proportion of variance retained.

### 6.2 External Metrics (when ground truth is available for validation)

- **Adjusted Rand Index (ARI)** — measures agreement between predicted and true clusters, adjusted for chance.
- **Normalized Mutual Information (NMI)** — information shared between predicted and true clusterings.

These require known labels, which defeats the purpose in production but is useful for benchmarking.

### 6.3 Downstream Evaluation

The most practical approach: use unsupervised output as input to a supervised task and measure if it improves performance. If PCA-reduced features improve classification accuracy, the reduction preserved useful information.

In [ ]:
# Comparing clustering algorithms with internal metrics
algorithms = {
    "K-Means": KMeans(n_clusters=4, random_state=42, n_init=10),
    "Agglomerative": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "GMM": GaussianMixture(n_components=4, random_state=42),
}

print(f"{'Algorithm':15s} {'Silhouette':>12s} {'Calinski-H':>12s}")
print("-" * 40)
for name, algo in algorithms.items():
    labels = algo.fit_predict(X)
    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    print(f"{name:15s} {sil:12.4f} {ch:12.1f}")

---

## 7. Choosing the Right Technique

| Goal | Technique | When to Use |
|------|-----------|-------------|
| Group similar items | K-Means | Spherical clusters, known K, large data |
| Group similar items | Hierarchical | Unknown K, need dendrogram, small-medium data |
| Group similar items | DBSCAN | Arbitrary shapes, unknown K, need outlier detection |
| Group similar items | GMM | Elliptical clusters, need probabilities |
| Reduce dimensions | PCA | Linear structure, preprocessing, variance preservation |
| Visualize data | t-SNE / UMAP | Nonlinear structure, 2D/3D visualization |
| Find unusual points | Isolation Forest | General-purpose, high-dimensional |
| Find unusual points | One-Class SVM | Well-defined normal class, smaller data |
| Find item relationships | Apriori / FP-Growth | Transactional/basket data |

Always **scale features** before clustering, PCA, or distance-based methods. Features on different scales will dominate distance calculations.

---

## 8. Unsupervised Learning in Practice

Unsupervised learning rarely stands alone in production systems. Typical roles:

**Exploratory analysis** — cluster customers to discover segments before designing marketing campaigns. Visualize high-dimensional data to understand structure.

**Preprocessing** — reduce dimensions with PCA before training a classifier. Cluster-based features (cluster membership as a new column) can improve supervised models.

**Anomaly monitoring** — continuously score incoming data for anomalies. Flag suspicious transactions, network events, or sensor readings for human review.

**Representation learning** — autoencoders and self-supervised methods learn compressed representations that capture essential data structure, used as features for downstream tasks.

The key mindset: unsupervised learning discovers structure you did not know to look for. The results require human interpretation — clusters must be named and understood, anomalies must be investigated, reduced dimensions must be validated.

In [ ]:
# PCA as preprocessing: does it help classification?
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

X_iris = StandardScaler().fit_transform(load_iris().data)
y_iris = load_iris().target

scores_full = cross_val_score(KNeighborsClassifier(5), X_iris, y_iris, cv=5)
X_pca_iris = PCA(n_components=2).fit_transform(X_iris)
scores_pca = cross_val_score(KNeighborsClassifier(5), X_pca_iris, y_iris, cv=5)

print(f"KNN on 4 features:  CV accuracy = {scores_full.mean():.4f} ± {scores_full.std():.4f}")
print(f"KNN on 2 PCA components: CV accuracy = {scores_pca.mean():.4f} ± {scores_pca.std():.4f}")
print("\nDimensionality reduction trades dimensions for variance —")
print("performance may drop, but computation is faster and visualization is possible.")

---

## 9. Summary

Unsupervised learning discovers structure in unlabeled data. **Clustering** (K-Means, hierarchical, DBSCAN, GMM) groups similar points. **Dimensionality reduction** (PCA, t-SNE) compresses data while preserving information. **Anomaly detection** (Isolation Forest, One-Class SVM) identifies unusual observations. **Association rule mining** finds relationships in transactional data.

Evaluation is challenging without labels — internal metrics, visualization, and downstream task performance provide guidance. Feature scaling is essential before most unsupervised algorithms.

Unsupervised learning powers exploratory analysis, preprocessing pipelines, and anomaly monitoring. Combined with supervised learning, it forms a complete toolkit for extracting value from data — whether the answers are known in advance or waiting to be discovered.